# MMA Fight Analyzer — complete Kaggle experiment, selection, and demo runner

This notebook reproduces the pipeline from raw labeled clips through preprocessing, gate evaluation, phase/pressure experiments, ablations, temporal smoothing, baseline architectures, final-model selection, and an annotated demo video.

Use a **Kaggle T4 ×2** accelerator with Internet enabled. The repository scripts remain the source of truth; this notebook only orchestrates them and displays their artifacts.

## Persistence and storage — read before running

Kaggle storage is local to the session. This workflow writes: `data/raw/` (downloaded MP4s), `data/cache/` (16-frame NPZ cache), `outputs/gate/`, `outputs/phase/`, `outputs/report/`, and `outputs/demo/`. Training is resumable fold-by-fold because every fold has a unique checkpoint and prediction file.

At the end, the notebook creates `mma_experiment_outputs.zip`. Download it from Kaggle's Output panel or use **Save Version**. Optionally create a private Kaggle Dataset from `mma_cache.zip`; on the next session attach it and set `RESTORE_CACHE_ARCHIVE`. Do not repeatedly decode and preprocess 1,315 videos if the cache already exists.

In [ ]:
from pathlib import Path
import json, os, shlex, shutil, subprocess, sys, time

WORK = Path('/kaggle/working')
REPO = WORK / 'mma-fight-analyzer'
GITHUB_REPO = 'https://github.com/Maximilianb1/mma-fight-analyzer.git'
DRIVE_FOLDER_ID = '1bYKlbqa-Mu7UKUIez7EfxA7xtqWvXwyI'
RESTORE_CACHE_ARCHIVE = None   # e.g. '/kaggle/input/mma-cache/mma_cache.zip'
RESTORE_OUTPUTS_ARCHIVE = None # e.g. '/kaggle/input/mma-results/mma_experiment_outputs.zip'
ARCHIVE_CACHE_AT_END = False   # True creates a large cache ZIP for reuse
RUN_SMALL_TUNING = True
FINAL_OVERRIDES = []  # after the pilot, optionally set e.g. ['--lr','1e-4']
print('Configuration loaded')

In [ ]:
# Confirm that Kaggle assigned two T4s. Do not use the P100 with the current Kaggle torch build.
subprocess.run(['nvidia-smi', '-L'], check=False)

## 1. Repository and environment

For the private GitHub repository, create a Kaggle secret named `GITHUB_TOKEN` containing a read-capable GitHub token. The token is passed directly to Git and is never printed. Kaggle already provides CUDA PyTorch, so only missing packages are installed.

In [ ]:
if not REPO.exists():
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        pass
    if not token:
        raise RuntimeError('Private repository: add a Kaggle secret named GITHUB_TOKEN, then rerun.')
    clone_url = GITHUB_REPO.replace('https://', f'https://x-access-token:{token}@')
    subprocess.run(['git', 'clone', clone_url, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'gdown'], check=True)
subprocess.run(['git', 'status', '--short', '--branch'], check=True)
print('Repository:', REPO)

In [ ]:
# Experiment helpers. Independent jobs are pinned to different GPUs; this is how T4 x2 is used.
PHASE_OUT = REPO / 'outputs/phase'
PHASE_OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR = REPO / 'outputs/logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    'r2plus1d': dict(model='r2plus1d', task='both', extra=[]),
    'r2plus1d_phaseonly': dict(model='r2plus1d', task='phase', extra=[]),
    'r2plus1d_pressureonly': dict(model='r2plus1d', task='pressure', extra=[]),
    'r2plus1d_nomask': dict(model='r2plus1d', task='both', extra=['--no-mask']),
    'r2plus1d_pressureonly_nomask': dict(model='r2plus1d', task='pressure', extra=['--no-mask']),
    'r2plus1d_pressureonly_hierarchical': dict(model='r2plus1d', task='pressure', extra=['--pressure-head','hierarchical']),
    'r3d': dict(model='r3d', task='both', extra=[]),
    'mc3': dict(model='mc3', task='both', extra=[]),
}

def command_for(tag, final=False, run_name=None, folds=None, overrides=None):
    spec = EXPERIMENTS[tag]
    cmd = ['scripts/train_phase.py', '--model', spec['model'], '--task', spec['task'],
           '--out', 'outputs/phase', '--run-name', run_name or tag, *spec['extra']]
    if final: cmd += ['--final']
    if folds is not None: cmd += ['--folds', str(folds)]
    if overrides: cmd += overrides
    return cmd

def start_job(name, command, gpu):
    log_path = LOG_DIR / f'{name}.log'
    env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES'] = str(gpu); env['PYTHONUNBUFFERED'] = '1'
    log = open(log_path, 'w', encoding='utf-8')
    proc = subprocess.Popen([sys.executable, *command], env=env, stdout=log, stderr=subprocess.STDOUT)
    return proc, log, log_path

def finish_job(job):
    proc, log, path = job; code = proc.wait(); log.close()
    tail = path.read_text(encoding='utf-8', errors='replace').splitlines()[-35:]
    print(f'--- {path.name} (exit={code}) ---'); print('\n'.join(tail))
    if code: raise RuntimeError(f'job failed; inspect {path}')

def run_experiment(tag, gpu=0, final=False, run_name=None, folds=None, overrides=None, force=False):
    artifact_tag = run_name or tag
    done = (PHASE_OUT / (f'{artifact_tag}_final.pt' if final else f'{artifact_tag}_metrics.json'))
    if done.exists() and not force:
        print('skip existing', done); return None
    job = start_job(artifact_tag, command_for(tag, final, run_name, folds, overrides), gpu)
    finish_job(job); return done

def run_pair(a, b, **kwargs):
    jobs = []
    for gpu, tag in enumerate((a, b)):
        done = PHASE_OUT / f'{tag}_metrics.json'
        if done.exists(): print('skip existing', done)
        else: jobs.append(start_job(tag, command_for(tag), gpu))
    for job in jobs: finish_job(job)

## 2. Restore/download data and preprocess once

`download_data.py` creates `data/raw/<fight>/`. `preprocess.py` creates one resumable `data/cache/<fight>/<clip>.npz` per clip, containing 16 RGB frames and the identity mask. Existing cache files are skipped.

In [ ]:
def restore(archive, destination):
    if archive and Path(archive).exists():
        destination.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archive), str(destination))
        print('restored', archive, '->', destination)

restore(RESTORE_CACHE_ARCHIVE, REPO)
restore(RESTORE_OUTPUTS_ARCHIVE, REPO)
if len(list((REPO / 'data/raw').glob('*/*.mp4'))) < 1315:
    subprocess.run([sys.executable, 'scripts/download_data.py', '--folder-id', DRIVE_FOLDER_ID], check=True)
cache_count = len(list((REPO / 'data/cache').glob('*/*.npz')))
if cache_count < 1315:
    env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES'] = '0'
    subprocess.run([sys.executable, 'scripts/preprocess.py'], env=env, check=True)
print('raw MP4s:', len(list((REPO/'data/raw').glob('*/*.mp4'))), 'cache files:', len(list((REPO/'data/cache').glob('*/*.npz'))))

In [ ]:
# Dataset statistics and a report-ready distribution figure.
sys.path.insert(0, str(REPO/'src'))
import matplotlib.pyplot as plt, pandas as pd, seaborn as sns
from mma.data import discover_clips
records_all = discover_clips(REPO/'data/raw', include_excluded=True)
records = records_all[~records_all.excluded].copy()
report_dir = REPO/'outputs/report'; report_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.countplot(data=records, y='phase_label', order=records.phase_label.value_counts().index, ax=axes[0], color='#4C72B0')
sns.countplot(data=records, y='pressure_label', order=records.pressure_label.value_counts().index, ax=axes[1], color='#DD8452')
pd.Series({'Fight': len(records), 'Non-fight': int(records_all.excluded.sum())}).plot.bar(ax=axes[2], color=['#55A868','#C44E52'])
axes[0].set_title('Phase labels'); axes[1].set_title('Pressure labels'); axes[2].set_title('Gate labels')
fig.suptitle(f'{len(records_all):,} clips across {records_all.fight.nunique()} fights')
fig.tight_layout(); fig.savefig(report_dir/'dataset_distribution.png', dpi=180); plt.show()
display(records_all.groupby('fight').agg(clips=('filename','size'), excluded=('excluded','sum')))

## 3. Train and evaluate the fight/non-fight gate

The gate selects its threshold from clip-level probabilities, matching inference (mean of four sampled frames). It saves frame/clip AUC, AP, precision, recall, F1, ROC/PR curves, a confusion matrix, predictions, and training history.

In [ ]:
gate_ckpt = REPO/'outputs/gate/gate.pt'
if not gate_ckpt.exists():
    job = start_job('gate', ['scripts/train_gate.py'], 0); finish_job(job)
gate_metrics = json.loads((REPO/'outputs/gate/gate_metrics.json').read_text())
display(pd.DataFrame([gate_metrics]).T.rename(columns={0:'value'}))
from IPython.display import Image, display as show
show(Image(filename=str(REPO/'outputs/gate/gate_curves.png')))
show(Image(filename=str(REPO/'outputs/gate/gate_confusion.png')))

## 4. Core experiment: multi-task vs two single-task models

Pressure-only and phase-only train concurrently on the two T4s. The standard multi-task run then provides the direct comparison. All use identical fight-level folds.

In [ ]:
run_pair('r2plus1d_pressureonly', 'r2plus1d_phaseonly')
run_experiment('r2plus1d', gpu=0)

In [ ]:
def collect_results():
    rows=[]
    for path in sorted(PHASE_OUT.glob('*_metrics.json')):
        r=json.loads(path.read_text()); rows.append({
            'tag': path.name[:-len('_metrics.json')],
            'phase_f1': r.get('phase',{}).get('macro_f1'), 'phase_acc': r.get('phase',{}).get('accuracy'),
            'pressure_f1': r.get('pressure',{}).get('macro_f1'), 'pressure_acc': r.get('pressure',{}).get('accuracy')})
    return pd.DataFrame(rows).sort_values('tag').reset_index(drop=True)
core = collect_results(); display(core)

## 5. Identity-mask ablation and hierarchical pressure

The no-mask ablation follows whichever original method currently has better pressure macro-F1. In parallel, hierarchical pressure factors the task into Mutual-vs-Directional and Fighter-1-vs-Fighter-2.

In [ ]:
core = collect_results().set_index('tag')
best_core_pressure = core.loc[['r2plus1d','r2plus1d_pressureonly'],'pressure_f1'].idxmax()
ablation_tag = 'r2plus1d_pressureonly_nomask' if best_core_pressure.endswith('pressureonly') else 'r2plus1d_nomask'
jobs=[]
for gpu, tag in enumerate((ablation_tag, 'r2plus1d_pressureonly_hierarchical')):
    if not (PHASE_OUT/f'{tag}_metrics.json').exists(): jobs.append(start_job(tag, command_for(tag), gpu))
for job in jobs: finish_job(job)
display(collect_results())

## 6. Temporal smoothing without retraining

Centered three-window probability averaging and majority voting are evaluated independently within each fight. This is an experiment, not an automatic improvement: retain it only if OOF macro-F1 rises.

In [ ]:
table = collect_results().set_index('tag')
best_phase_tag = table.phase_f1.dropna().idxmax()
subprocess.run([sys.executable, 'scripts/temporal_smoothing.py', '--tag', best_phase_tag, '--task', 'phase', '--window', '3'], check=True)
smooth = json.loads((PHASE_OUT/f'smoothing_{best_phase_tag}.json').read_text())
display(pd.DataFrame(smooth['tasks']['phase']).T[['accuracy','macro_f1']])
show(Image(filename=str(PHASE_OUT/f'smoothing_phase_{best_phase_tag}.png')))

## 7. Easy pretrained video baselines

R3D-18 and MC3-18 are Kinetics-pretrained drop-in baselines using the same input, identity channel, heads, losses, augmentation, and fight-level folds. They train concurrently.

In [ ]:
run_pair('r3d', 'mc3')

In [ ]:
# Final comparison table and report-ready chart.
results = collect_results(); results.to_csv(report_dir/'experiment_summary.csv', index=False)
display(results.sort_values('phase_f1', ascending=False, na_position='last'))
fig, axes = plt.subplots(1,2,figsize=(15,5))
for ax, task in zip(axes, ('phase','pressure')):
    d=results.dropna(subset=[f'{task}_f1']).sort_values(f'{task}_f1')
    ax.barh(d.tag, d[f'{task}_f1'], color='#4C72B0' if task=='phase' else '#DD8452')
    ax.set_xlim(0,1); ax.set_title(f'{task.title()} macro-F1 (OOF)')
fig.tight_layout(); fig.savefig(report_dir/'all_experiments.png', dpi=180); plt.show()

## 8. Small, transparent hyperparameter pilot

This is deliberately limited to fold 0: one lower learning rate and one stronger pressure loss. Treat it as exploratory tuning, not as an independent final result. Only rerun all folds for a setting if the pilot improvement is convincing.

In [ ]:
if RUN_SMALL_TUNING:
    tuning = [
        ('tune_lr1e4', ['--lr','1e-4']),
        ('tune_pressure1', ['--pressure-weight','1.0']),
    ]
    jobs=[]
    for gpu,(name,overrides) in enumerate(tuning):
        pred=PHASE_OUT/f'{name}_fold0_preds.npz'
        if not pred.exists(): jobs.append(start_job(name, command_for('r2plus1d', run_name=name, folds=0, overrides=overrides), gpu))
    for job in jobs: finish_job(job)
    from sklearn.metrics import f1_score
    tuning_rows=[]
    for name,_ in tuning:
        d=dict(__import__('numpy').load(PHASE_OUT/f'{name}_fold0_preds.npz'))
        ph=f1_score(d['phase_true'],d['phase_pred'],average='macro',zero_division=0)
        pr=f1_score(d['pressure_true'],d['pressure_pred'],average='macro',zero_division=0)
        tuning_rows.append({'pilot':name,'phase_f1':ph,'pressure_f1':pr,'combined':ph+.5*pr})
    display(pd.DataFrame(tuning_rows).sort_values('combined',ascending=False))

## 9. Select winners and train deployment checkpoint(s) on all fights

Selection uses OOF macro-F1 from complete four-fold runs. If one multi-task experiment wins both tasks, one final checkpoint is trained. Otherwise the best phase and pressure configurations train concurrently and inference combines them.

In [ ]:
results = collect_results().set_index('tag')
eligible = results.loc[results.index.intersection(EXPERIMENTS)]
best_phase_tag = eligible.phase_f1.dropna().idxmax()
best_pressure_tag = eligible.pressure_f1.dropna().idxmax()
print('Best phase:', best_phase_tag, 'Best pressure:', best_pressure_tag)
if best_phase_tag == best_pressure_tag and EXPERIMENTS[best_phase_tag]['task'] == 'both':
    run_experiment(best_phase_tag, gpu=0, final=True, run_name='deployment_combined', overrides=FINAL_OVERRIDES)
    FINAL_PHASE_CKPT = PHASE_OUT/'deployment_combined_final.pt'; FINAL_PRESSURE_CKPT = None
else:
    jobs=[]
    for gpu,(tag,name) in enumerate(((best_phase_tag,'deployment_phase'),(best_pressure_tag,'deployment_pressure'))):
        path=PHASE_OUT/f'{name}_final.pt'
        if not path.exists(): jobs.append(start_job(name, command_for(tag, final=True, run_name=name, overrides=FINAL_OVERRIDES), gpu))
    for job in jobs: finish_job(job)
    FINAL_PHASE_CKPT = PHASE_OUT/'deployment_phase_final.pt'
    FINAL_PRESSURE_CKPT = PHASE_OUT/'deployment_pressure_final.pt'
print('Phase checkpoint:', FINAL_PHASE_CKPT); print('Pressure checkpoint:', FINAL_PRESSURE_CKPT)

## 10. End-to-end demo

Attach a raw full fight/video as a Kaggle Dataset and set the path and fighter metadata below. Headless Kaggle uses shorts colors instead of the interactive A/B prompt. The output is an annotated MP4 plus a per-window JSON log.

In [ ]:
DEMO_VIDEO = None  # e.g. Path('/kaggle/input/mma-demo/demo_fight.mp4')
F1_NAME, F2_NAME = 'Fighter 1', 'Fighter 2'
F1_COLOR, F2_COLOR = 'black', 'red'
demo_dir=REPO/'outputs/demo'; demo_dir.mkdir(parents=True,exist_ok=True)
if DEMO_VIDEO and Path(DEMO_VIDEO).exists():
    demo_out=demo_dir/'fight_labeled.mp4'
    cmd=['scripts/infer.py','--video',str(DEMO_VIDEO),'--out',str(demo_out),
         '--gate-ckpt',str(REPO/'outputs/gate/gate.pt'),'--phase-ckpt',str(FINAL_PHASE_CKPT),
         '--f1-name',F1_NAME,'--f2-name',F2_NAME,'--f1-color',F1_COLOR,'--f2-color',F2_COLOR]
    if FINAL_PRESSURE_CKPT: cmd += ['--pressure-ckpt',str(FINAL_PRESSURE_CKPT)]
    job=start_job('demo',cmd,0); finish_job(job)
    from IPython.display import Video
    display(Video(str(demo_out),embed=True,width=900))
    display(pd.read_json(demo_out.with_suffix('.json')).head(20))
else:
    print('Set DEMO_VIDEO and fighter names/colors, then rerun this cell.')

## 11. Package everything before ending the session

The outputs ZIP contains checkpoints, predictions, histories, metrics, plots, report tables, logs, and the demo. The optional cache ZIP is large but prevents preprocessing in the next session.

In [ ]:
outputs_zip=shutil.make_archive(str(WORK/'mma_experiment_outputs'),'zip',root_dir=REPO,base_dir='outputs')
print('Download/save:',outputs_zip, f'{Path(outputs_zip).stat().st_size/1e9:.2f} GB')
if ARCHIVE_CACHE_AT_END:
    cache_zip=shutil.make_archive(str(WORK/'mma_cache'),'zip',root_dir=REPO,base_dir='data/cache')
    print('Cache archive:',cache_zip, f'{Path(cache_zip).stat().st_size/1e9:.2f} GB')
print('Before closing Kaggle: inspect the demo, download the ZIP or Save Version, and record the winning tags in the report.')

## Report checklist produced by this notebook

- Dataset distributions and fight-grouped protocol
- Gate ROC-AUC, AP, F1, operating threshold, ROC/PR curves, and confusion matrix
- Multi-task vs phase-only vs pressure-only comparison
- Identity-mask ablation
- Flat vs hierarchical pressure head
- R(2+1)D vs R3D vs MC3 pretrained video architectures
- Per-class confusion matrices, per-fight metrics, and fold training curves
- Raw vs temporal-smoothed OOF metrics
- Transparent small hyperparameter pilot
- Final full-data checkpoint(s), annotated demo video, and per-window log

Add the separately collected inter-annotator Cohen's kappa results to complete the label-quality analysis.